In [1]:
import pandas as pd

In [2]:
df=pd.read_csv('../data/sales_clean.csv',parse_dates=['InvoiceDate'])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1007910 entries, 0 to 1007909
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1007910 non-null  int64         
 1   StockCode    1007910 non-null  str           
 2   Description  1007910 non-null  str           
 3   Quantity     1007910 non-null  int64         
 4   InvoiceDate  1007910 non-null  datetime64[us]
 5   Price        1007910 non-null  float64       
 6   Customer ID  779423 non-null   float64       
 7   Country      1007910 non-null  str           
 8   Revenue      1007910 non-null  float64       
dtypes: datetime64[us](1), float64(3), int64(2), str(3)
memory usage: 69.2 MB


In [3]:
missing = df["Customer ID"].isna().sum()
print(f"Rows without customers: {missing} ({missing/len(df):.1%})")
print(df["InvoiceDate"].min(),df["InvoiceDate"].max())
print(df["Customer ID"].nunique())

Rows without customers: 228487 (22.7%)
2009-12-01 07:45:00 2011-12-09 12:50:00
5878


In [4]:
df=df.dropna(subset=["Customer ID"]) 
df['Customer ID']=df['Customer ID'].astype(int)   
print(df.shape,df['Customer ID'].nunique())

(779423, 9) 5878


In [5]:
cutoff=pd.Timestamp("2011-09-09")
windows_end= cutoff +pd.Timedelta(days=90)

history = df[df['InvoiceDate']<cutoff]
window = df[(df['InvoiceDate']>=cutoff) & (df['InvoiceDate']<windows_end)]

print(history['InvoiceDate'].max())
print(window['InvoiceDate'].min(), window["InvoiceDate"].max())

2011-09-08 19:58:00
2011-09-09 08:48:00 2011-12-07 17:25:00


In [6]:
bought_in_window = window['Customer ID']
order_counts = history.groupby('Customer ID')['Invoice'].nunique()
print(order_counts.head())

Customer ID
12346    11
12347     6
12348     4
12349     3
12350     1
Name: Invoice, dtype: int64


In [7]:
regular_counts = order_counts[order_counts>=5]
regular_customers = regular_counts.index
print(len(regular_customers))



1798


In [8]:
churned = (~regular_customers.isin(bought_in_window)).astype(int)
print(churned[:5],churned.sum(),len(churned))

[1 0 0 0 0] 539 1798


In [9]:
labels = pd.DataFrame({"churned":churned},index=regular_customers)
labels.head()


,churned
Customer ID,
12346,1
12347,0
12352,0
12356,0
12359,0


In [10]:
print(labels.shape, labels["churned"].mean())

(1798, 1) 0.29977753058954393


In [11]:
history.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [12]:
last_order=history.groupby("Customer ID")["InvoiceDate"].max()
recency = (cutoff - last_order).dt.days
print(recency.head())
print(recency.min(),recency.max())


Customer ID
12346    437
12347     37
12348    156
12349    315
12350    218
Name: InvoiceDate, dtype: int64
0 646


In [13]:
recency = recency.loc[regular_customers]
recency.shape


(1798,)

In [14]:
print(recency.min(),recency.max())

0 599


In [15]:
frequency = order_counts.loc[regular_customers]
print(frequency.shape,frequency.min())

(1798,) 5


In [16]:
features = pd.DataFrame({"recency":recency,"frequency":frequency})
print(features.head())
print(features.shape)
print(features.isna().sum())

             recency  frequency
Customer ID                    
12346            437         11
12347             37          6
12352            170          7
12356            153          5
12359             97          9
(1798, 2)
recency      0
frequency    0
dtype: int64


In [17]:
monetary = history.groupby('Customer ID')['Revenue'].sum()
monetary = monetary.loc[regular_customers]
print(monetary.head())
print(f"the shape is {monetary.shape}")


Customer ID
12346     372.86
12347    3402.39
12352    1905.61
12356    6313.38
12359    6059.09
Name: Revenue, dtype: float64
the shape is (1798,)


In [18]:
features['monetary']=monetary
print(features.shape)
print(f"the number of null values are:{features['monetary'].isna().sum()}")

(1798, 3)
the number of null values are:0


In [19]:
features['avg_order']= features['monetary']/features['frequency']
print(features['avg_order'].describe())

count    1798.000000
mean      407.694422
std       409.176985
min        11.161333
25%       214.786979
50%       316.499857
75%       454.704712
max      6819.052000
Name: avg_order, dtype: float64


In [20]:
features.head()

,recency,frequency,monetary,avg_order
Customer ID,,,,
12346,437,11,372.86,33.896364
12347,37,6,3402.39,567.065000
12352,170,7,1905.61,272.230000
12356,153,5,6313.38,1262.676000
12359,97,9,6059.09,673.232222


In [21]:
features.shape

(1798, 4)

In [25]:
variety = history.groupby("Customer ID")['StockCode'].nunique()
variety= variety.loc[regular_customers]
variety.head()

Customer ID
12346     26
12347    107
12352     38
12356    104
12359    230
Name: StockCode, dtype: int64

In [35]:
features['variety']=variety
print(f"The sum of null values is: {features['variety'].isna().sum()}")
print(features.shape)

The sum of null values is: 0
(1798, 5)


In [36]:
history.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [42]:
orders = history.groupby(["Customer ID", "Invoice"])["InvoiceDate"].min().reset_index()
print(orders.head())
print(orders.shape)

   Customer ID  Invoice         InvoiceDate
0        12346   491725 2009-12-14 08:34:00
1        12346   491742 2009-12-14 11:00:00
2        12346   491744 2009-12-14 11:02:00
3        12346   492718 2009-12-18 10:47:00
4        12346   492722 2009-12-18 10:55:00
(30284, 3)


In [46]:
orders = orders.sort_values(["Customer ID","InvoiceDate"])
orders['gap'] = orders.groupby("Customer ID")['InvoiceDate'].diff()
orders.head()
                            



            

,Customer ID,Invoice,InvoiceDate,gap
0,12346,491725,2009-12-14 08:34:00,NaT
1,12346,491742,2009-12-14 11:00:00,0 days 02:26:00
2,12346,491744,2009-12-14 11:02:00,0 days 00:02:00
3,12346,492718,2009-12-18 10:47:00,3 days 23:45:00
4,12346,492722,2009-12-18 10:55:00,0 days 00:08:00


In [50]:
orders['gap_days']= orders['gap'].dt.days
usual_gap = orders.groupby("Customer ID")['gap_days'].mean()
usual_gap = usual_gap.loc[regular_customers]
features["usual_gap"] = usual_gap
print(features.shape)
print(features["usual_gap"].isna().sum())
print(features["usual_gap"].describe())

(1798, 6)
0
count    1798.000000
mean       56.269595
std        31.033516
min         1.707792
25%        32.729167
50%        51.500000
75%        75.851190
max       157.500000
Name: usual_gap, dtype: float64


In [52]:
recent_start = cutoff - pd.Timedelta(days=90)
recent_start

Timestamp('2011-06-11 00:00:00')

In [54]:
recent = history[history['InvoiceDate']>=recent_start]
print(recent.shape,history.shape)

(78957, 9) (619307, 9)


In [55]:
recent_orders= recent.groupby("Customer ID")['Invoice'].nunique()
recent_orders

Customer ID
12347    1
12358    1
12360    1
12362    2
12363    1
        ..
18272    2
18273    1
18281    1
18282    1
18283    5
Name: Invoice, Length: 1892, dtype: int64

In [59]:
features['recent_orders']=recent_orders
features['recent_orders']=features['recent_orders'].fillna(0)
print(features.shape)
print(features["recent_orders"].isna().sum())
print((features["recent_orders"] == 0).sum())
print(features["recent_orders"].describe())

(1798, 7)
0
635
count    1798.000000
mean        1.665184
std         2.729142
min         0.000000
25%         0.000000
50%         1.000000
75%         2.000000
max        46.000000
Name: recent_orders, dtype: float64


In [60]:
features

,recency,frequency,monetary,avg_order,variety,usual_gap,recent_orders
Customer ID,,,,,,,
12346,437,11,372.86,33.896364,26,19.200000,0.0
12347,37,6,3402.39,567.065000,107,54.400000,1.0
12352,170,7,1905.61,272.230000,38,21.500000,0.0
12356,153,5,6313.38,1262.676000,104,44.250000,0.0
12359,97,9,6059.09,673.232222,230,67.750000,0.0
...,...,...,...,...,...,...,...
18268,42,6,1490.23,248.371667,36,118.600000,1.0
18272,21,7,3413.85,487.692857,123,88.166667,2.0
18276,300,5,1320.66,264.132000,69,83.500000,0.0


In [61]:
features['recent_share']=features['recent_orders']/features['frequency']
print(features["recent_share"].describe())

count    1798.000000
mean        0.126266
std         0.139703
min         0.000000
25%         0.000000
50%         0.111111
75%         0.200000
max         1.000000
Name: recent_share, dtype: float64


In [71]:
recent_spend = recent.groupby("Customer ID")['Revenue'].sum()
features['recent_spend']= recent_spend
features['recent_spend']=features['recent_spend'].fillna(0)
features['spend_share']=features['recent_spend'] / features['monetary']
print(features.shape)
print(features.isna().sum())
print(features['spend_share'].describe())

(1798, 10)
recency          0
frequency        0
monetary         0
avg_order        0
variety          0
usual_gap        0
recent_orders    0
recent_share     0
recent_spend     0
spend_share      0
dtype: int64
count    1798.000000
mean        0.121349
std         0.140854
min         0.000000
25%         0.000000
50%         0.094802
75%         0.184220
max         1.000000
Name: spend_share, dtype: float64


In [72]:
data=features.join(labels)
print(data.shape)
print(data.isna().sum().sum())
print(data['churned'].mean())

(1798, 11)
0
0.29977753058954393


In [73]:
data.head()

,recency,frequency,monetary,avg_order,variety,usual_gap,recent_orders,recent_share,recent_spend,spend_share,churned
Customer ID,,,,,,,,,,,
12346,437,11,372.86,33.896364,26,19.20,0.0,0.000000,0.00,0.000000,1
12347,37,6,3402.39,567.065000,107,54.40,1.0,0.166667,584.91,0.171912,0
12352,170,7,1905.61,272.230000,38,21.50,0.0,0.000000,0.00,0.000000,0
12356,153,5,6313.38,1262.676000,104,44.25,0.0,0.000000,0.00,0.000000,0
12359,97,9,6059.09,673.232222,230,67.75,0.0,0.000000,0.00,0.000000,0


In [75]:
data.to_csv("../data/model_data.csv")
print(pd.read_csv("../data/model_data.csv", index_col=0).shape)

(1798, 11)
